In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import json, time, random
from datasets import load_dataset

from bait.utils import common_utils, file_utils, json_utils, container_utils, model_utils, tokenizer_utils

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data/raw_datasets'

In [ ]:
def download(save_dir, dataset_name, config_name=None):
    print(f"[start] {dataset_name} ({config_name if config_name else 'base'}) download..")

    if config_name:
        dataset = load_dataset(dataset_name, config_name, trust_remote_code=True)
        prefix = f"{config_name}_"
    else:
        dataset = load_dataset(dataset_name, trust_remote_code=True)
        prefix = ""
    
    for split_name, dataset_split in dataset.items():
        file_path = f'{save_dir}/{prefix}{split_name}.json'
        file_utils.make_parent(file_path)

        dataset_split_list = dataset_split.to_list()
        json_utils.write_json(dataset_split_list, file_path)
    print()

In [ ]:
# 멀티홉 QA 데이터셋
save_dir = f'{data_dir}/Multi-Hop_QA'

# 1. 2WikiQA (2WikiMultihopQA)
# https://github.com/Alab-NII/2wikimultihop
# download(f'{save_dir}/2WikiMultihopQA', 'xanhho/2WikiMultihopQA') -> 공식 경로 다운로드 안됨 -> 수동 다운로드함

# 2. HotpotQA
# https://hotpotqa.github.io/
download(f'{save_dir}/HotpotQA', 'hotpot_qa', 'distractor')
download(f'{save_dir}/HotpotQA', 'hotpot_qa', 'fullwiki')

# 3. MuSiQue
# https://github.com/stonybrooknlp/musique
download(f'{save_dir}/MuSiQue', 'bdsaglam/musique')

# 4. Bamboogle
# https://github.com/ofirpress/self-ask/tree/main
download(f'{save_dir}/Bamboogle', 'chiayewken/bamboogle')

In [ ]:
# 지식 충돌 QA 데이터셋
save_dir = f'{data_dir}/Knowledge-Conflict_QA'

# 1. CoConflictQA (ParamMute)
# https://github.com/OpenBMB/ParamMute
download(f'{save_dir}/CoConflictQA', 'chengpingan/CoConflictQA')

# 2. StanfordClashEval
# https://github.com/kevinwu23/StanfordClashEval
# 원본은 kewu93/ClashEval 인데, 삭제됨
download(f'{save_dir}/StanfordClashEval', 'sagnikrayc/clasheval')

In [ ]:
in_file_paths = file_utils.get_file_paths(data_dir, True)

for in_file_path in in_file_paths:
    file_dir = os.path.dirname(in_file_path)
    file_name = file_utils.get_file_name(in_file_path)

    if file_name.startswith('sample_'):
        continue

    dataset = json_utils.load_json(in_file_path)

    if len(dataset) >= 200:
        sample_dataset = dataset[:100] + random.sample(dataset[100:], 100)
    else:
        sample_dataset = dataset
        
    sample_file_path = f'{file_dir}/sample_{file_name}'
    json_utils.write_json(sample_dataset, sample_file_path, do_print=False)